# 04: Topic-Level Seasonal Differences
This notebook examines whether different knowledge domains (Education, Politics, Entertainment) exhibit distinct seasonal patterns in Wikipedia pageviews using **page clusters** and **min-max normalization**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import numpy as np
from scipy.stats import pearsonr

# Add src to path
sys.path.append(os.path.abspath('../../'))
from src.data_prep import clean_pageview_data, add_time_features

# Configuration
DATA_PATH = '../../data/raw/en_wiki_topic_pageviews_daily.csv'
GLOBAL_DATA_PATH = '../../data/raw/en_wiki_pageviews_daily.csv'
REPORT_DIR = '../../reports/'
os.makedirs(REPORT_DIR, exist_ok=True)

sns.set_theme(style="whitegrid")

## 1. Load Topic Data

In [ ]:
df_topics = pd.read_csv(DATA_PATH)
df_topics['timestamp'] = pd.to_datetime(df_topics['timestamp'])
df_topics = add_time_features(df_topics)

# Ensure correct weekday ordering
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df_topics['day_of_week'] = pd.Categorical(df_topics['day_of_week'], categories=day_order, ordered=True)

print(f"Loaded {len(df_topics)} records for {df_topics['article'].nunique()} articles in {df_topics['cluster'].nunique()} clusters.")
df_topics.head()

## 2. Methodology

### 2.1 Seasonal Shape Comparison (Min-Max Normalization)
Normalize each topic's traffic independently to compare seasonal patterns regardless of absolute volume.

In [ ]:
def normalize_series(group):
    min_v, max_v = group['views'].min(), group['views'].max()
    group['norm_views'] = (group['views'] - min_v) / (max_v - min_v) if max_v != min_v else 0
    return group

df_topics = df_topics.groupby('article', group_keys=False).apply(normalize_series)

# Aggregate to Cluster Level
cluster_ts = df_topics.groupby(['cluster', 'timestamp'])[['views', 'norm_views']].mean().reset_index()
cluster_ts = add_time_features(cluster_ts)

# Plot Seasonal Shape (Month-wise Average Intensity)
seasonal_profile = cluster_ts.groupby(['cluster', 'month'])['norm_views'].mean().reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(data=seasonal_profile, x='month', y='norm_views', hue='cluster', marker='o', linewidth=2.5)
plt.title('Normalized Seasonal Shape by Topic Category', fontsize=15)
plt.xlabel('Month', fontsize=12)
plt.ylabel('Normalized Traffic Intensity', fontsize=12)
plt.xticks(range(1, 13))
plt.legend(title='Topic Cluster')
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'section_4_topic_seasonal_shape.png'))
plt.show()

### 2.2 Seasonal Amplitude Measurement
Measure the strength of fluctuation: $Amplitude = \frac{Peak - Min}{Mean}$.

In [ ]:
amplitude_stats = cluster_ts.groupby('cluster')['views'].agg(['max', 'min', 'mean']).reset_index()
amplitude_stats['amplitude'] = (amplitude_stats['max'] - amplitude_stats['min']) / amplitude_stats['mean']

plt.figure(figsize=(10, 6))
sns.barplot(data=amplitude_stats, x='cluster', y='amplitude', palette='viridis')
plt.title('Seasonal Amplitude (Peak-to-Trough) by Category', fontsize=15)
plt.ylabel('Amplitude Score', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'section_4_topic_amplitude.png'))
plt.show()

print(amplitude_stats[['cluster', 'amplitude']])

### 2.3 Seasonal Stability Test
Calculate correlation between seasonal profiles across consecutive years to test consistency.

In [ ]:
def calculate_stability(df):
    years = sorted(df['year'].unique())
    correlations = []
    
    for cluster in df['cluster'].unique():
        cluster_df = df[df['cluster'] == cluster]
        profiles = []
        for year in years:
            # Get monthly profile for the year
            profile = cluster_df[cluster_df['year'] == year].groupby('month')['norm_views'].mean()
            if len(profile) == 12:
                profiles.append((year, profile.values))
        
        # Compare consecutive years
        for i in range(len(profiles) - 1):
            year1, p1 = profiles[i]
            year2, p2 = profiles[i+1]
            corr, _ = pearsonr(p1, p2)
            correlations.append({'cluster': cluster, 'year_pair': f"{year1}-{year2}", 'correlation': corr})
            
    return pd.DataFrame(correlations)

stability_df = calculate_stability(cluster_ts)
avg_stability = stability_df.groupby('cluster')['correlation'].mean().reset_index()

print("Average Seasonal Stability (Year-over-Year Correlation):")
print(avg_stability)

### 2.4 Lead–Lag Analysis (Cross-Correlation)
Check if spikes in one category tend to lead or lag another.

In [ ]:
def cross_corr(s1, s2, lag=0):
    return s1.corr(s2.shift(lag))

# Compare Education vs Entertainment
edu_series = cluster_ts[cluster_ts['cluster'] == 'education'].set_index('timestamp')['norm_views']
ent_series = cluster_ts[cluster_ts['cluster'] == 'entertainment'].set_index('timestamp')['norm_views']

lags = range(-30, 31)
corrs = [cross_corr(edu_series, ent_series, l) for l in lags]

plt.figure(figsize=(12, 6))
plt.plot(lags, corrs, color='purple', marker='o', markersize=4)
plt.axvline(x=0, color='red', linestyle='--')
plt.title('Cross-Correlation: Education vs Entertainment (Daily Lags)', fontsize=15)
plt.xlabel('Lag (Days)', fontsize=12)
plt.ylabel('Correlation', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'section_4_lead_lag_analysis.png'))
plt.show()

### 2.5 Topic Volatility Analysis
30-day rolling standard deviation of normalized traffic.

In [ ]:
vol_list = []
for c in cluster_ts['cluster'].unique():
    subset = cluster_ts[cluster_ts['cluster'] == c].copy().sort_values('timestamp')
    subset['rolling_std'] = subset['norm_views'].rolling(window=30).std()
    vol_list.append(subset)

vol_df = pd.concat(vol_list)

plt.figure(figsize=(14, 7))
sns.lineplot(data=vol_df, x='timestamp', y='rolling_std', hue='cluster', alpha=0.6)
plt.title('Normalized Traffic Volatility (30-Day Rolling Std)', fontsize=15)
plt.ylabel('Volatility', fontsize=12)
plt.legend(title='Topic Cluster')
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'section_4_topic_volatility.png'))
plt.show()

### 2.6 Pandemic Resilience Comparison
Measure recovery time from the 2020 shock.

In [ ]:
def calculate_recovery_time(df, baseline_end='2020-02-01', shock_start='2020-03-01'):
    results = {}
    for c in df['cluster'].unique():
        subset = df[df['cluster'] == c].sort_values('timestamp')
        # Baseline: Average views in the 6 months before pandemic
        baseline_avg = subset[(subset['timestamp'] >= '2019-08-01') & (subset['timestamp'] < baseline_end)]['views'].mean()
        
        # Post-shock data
        post_shock = subset[subset['timestamp'] >= shock_start]
        
        # Find first date where monthly mean returns to baseline
        post_shock['monthly_mean'] = post_shock['views'].rolling(window=30).mean()
        recovery_point = post_shock[post_shock['monthly_mean'] <= baseline_avg]
        
        if not recovery_point.empty:
            recovery_date = recovery_point.iloc[0]['timestamp']
            months = (recovery_date.year - 2020) * 12 + (recovery_date.month - 3)
            results[c] = months
        else:
            results[c] = "Never Fully Recovered to Baseline"
            
    return results

recovery_metrics = calculate_recovery_time(cluster_ts)
print("Pandemic Recovery Time (Months after March 2020):")
for c, v in recovery_metrics.items():
    print(f"{c}: {v}")